# 02 · Tiendas DENUE — canal Moderno vs Tradicional (ciudad según `src/config.py`)

* **462111** – Comercio al por menor en supermercados
* **462112** – Comercio al por menor en minisupers
* **461110** – Comercio al por menor en tiendas de abarrotes, ultramarinos y misceláneas

**Canal:** *Moderno* = tienda de una cadena identificada (OXXO, Dunosusa, Willys, Aki, Walmart…).
*Tradicional* = establecimientos independientes (abarrotes, misceláneas y minisupers sin cadena).
Se entregan en **dos archivos xlsx** separados.

**Fuente oficial:** INEGI, DENUE – descarga masiva por entidad (edición vigente, se lee del
archivo de metadatos; verificada el 2026-09-24: edición 05_2026, publicada 2026-05-20).
* Página oficial: https://www.inegi.org.mx/app/descarga/?ti=6
* URL de descarga: `https://www.inegi.org.mx/contenidos/masiva/denue/denue_{ENT}_csv.zip` (en `src/config.py` → `FUENTES`)

Se identifica la **cadena** por nombre / razón social y se marcan las
tiendas **nuevas** (dadas de alta en el DENUE en los últimos 24 meses).

**Fuente complementaria (opcional):** OpenStreetMap (Overpass API) para detectar tiendas que aún no
aparecen en DENUE (p. ej. aperturas recientes). Solo se usa como alerta, no se mezcla con DENUE.
* Overpass API: https://overpass-api.de/api/interpreter (respaldo: https://overpass.kumi.systems/api/interpreter)

In [1]:
import re, sys, zipfile, io
from pathlib import Path

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())   # raíz del repo
sys.path.insert(0, str(BASE / "src"))
import numpy as np
import pandas as pd
import requests

import config as C
import cadenas
from descargas import descargar, registrar_fuentes, verificar_fuentes
from IPython.display import display

pd.set_option("display.width", 200, "display.max_columns", 30)


# reproducibilidad: ciudad, versiones y semilla (no hay pasos aleatorios en este notebook)
import platform
print(f"Repo: {BASE} | ciudad: {C.ZM_NOMBRE} (CIUDAD={C.CIUDAD}) | Python {platform.python_version()} · pandas {pd.__version__} · numpy {np.__version__}")
print("Nota: el cruce con OpenStreetMap (sección 3) consulta un servicio externo que cambia con el tiempo; "
      "solo genera alertas y no altera las tiendas DENUE ni los entregables.")

Repo: C:\Users\KIN\Desktop\Kin\mx-retail-geodemographics | ciudad: ZM Guadalajara (CIUDAD=guadalajara) | Python 3.14.6 · pandas 3.0.5 · numpy 2.5.1
Nota: el cruce con OpenStreetMap (sección 3) consulta un servicio externo que cambia con el tiempo; solo genera alertas y no altera las tiendas DENUE ni los entregables.


## 1. Descarga DENUE del estado

In [2]:
print(f"Ciudad: {C.ZM_NOMBRE} · {C.NOM_ENT} ({C.ENT}) · municipios: " + ", ".join(f"{v} ({k})" for k, v in C.ZM_MUNICIPIOS.items()))
display(verificar_fuentes(C.FUENTES, ["denue"]))
print("denue:", C.URLS["denue"])
descargar(C.URLS["denue"], C.ARCHIVOS["denue"])
with zipfile.ZipFile(C.ARCHIVOS["denue"]) as z:
    meta = z.read(next(n for n in z.namelist() if n.startswith("metadatos"))).decode("utf-8-sig", "replace")
    f_csv = next(n for n in z.namelist() if n.startswith("conjunto_de_datos") and n.endswith(".csv"))
    denue = pd.read_csv(z.open(f_csv), dtype=str, encoding="latin-1")
EDICION = re.search(r"Title:.*\(DENUE\)\s*(\S+)", meta).group(1)
print("Edición DENUE:", EDICION, f"| establecimientos en {C.NOM_ENT}:", f"{len(denue):,}")
registrar_fuentes(C.FUENTES, ["denue"], C.PROC / "fuentes_usadas_02.csv", extra={"denue": {"edicion_leida": EDICION}})

Ciudad: ZM Guadalajara · Jalisco (14) · municipios: Acatlán de Juárez (002), Guadalajara (039), Ixtlahuacán de los Membrillos (044), Juanacatlán (051), El Salto (070), Tlajomulco de Zúñiga (097), San Pedro Tlaquepaque (098), Tonalá (101), Zapopan (120), Zapotlanejo (124)


,ciudad,cobertura,nombre,edición,estado,MB,última modificación (servidor),página oficial,url de descarga
fuente,,,,,,,,,
denue,ZM Guadalajara,Jalisco (14) → se filtran los 10 municipios de...,Directorio Estadístico Nacional de Unidades Ec...,vigente (la edición exacta se lee de metadatos...,OK,39.4,"Wed, 20 May 2026 12:04:03 GMT",https://www.inegi.org.mx/app/descarga/?ti=6,https://www.inegi.org.mx/contenidos/masiva/den...


denue: https://www.inegi.org.mx/contenidos/masiva/denue/denue_14_csv.zip


[ok] ya existe denue_14_csv.zip


Edición DENUE: 05_2026 | establecimientos en Jalisco: 401,813


,fuente,ciudad,cobertura,nombre,edicion,url,pagina_oficial,archivo_local,bytes,descargado,sha256,edicion_leida
0,denue,ZM Guadalajara,Jalisco (14) → se filtran los 10 municipios de...,Directorio Estadístico Nacional de Unidades Ec...,vigente (la edición exacta se lee de metadatos...,https://www.inegi.org.mx/contenidos/masiva/den...,https://www.inegi.org.mx/app/descarga/?ti=6,C:\Users\KIN\Desktop\Kin\mx-retail-geodemograp...,39432220,2026-09-23T09:50:05,5a3ee180edecaa85be195c0efaa1bd918ffbd3dfb406fa...,05_2026


## 2. Filtro SCIAN + ZM y limpieza

In [3]:
t = denue[denue.codigo_act.isin(C.SCIAN_TIENDAS) & denue.cve_mun.isin(C.ZM_MUNICIPIOS)].copy()
for c in ["nom_estab", "raz_social", "localidad", "municipio", "nomb_asent", "nom_vial"]:
    t[c] = t[c].fillna("").str.strip()
t["latitud"] = pd.to_numeric(t.latitud, errors="coerce")
t["longitud"] = pd.to_numeric(t.longitud, errors="coerce")
t = t.dropna(subset=["latitud", "longitud"])
t["formato"] = t.codigo_act.map(C.SCIAN_TIENDAS)

# cadena y canal: patrones en src/cadenas.py (única fuente, compartida con los PDV de Bepensa)
t[["cadena", "canal"]] = cadenas.clasificar(t.nom_estab + " | " + t.raz_social)

# tamaño de establecimiento y antiguedad en DENUE
t["fecha_alta"] = pd.to_datetime(t.fecha_alta, format="%Y-%m", errors="coerce")
corte = t.fecha_alta.max() - pd.DateOffset(months=24)
t["nueva_24m"] = t.fecha_alta >= corte

tiendas = t[["id", "clee", "nom_estab", "raz_social", "codigo_act", "formato", "canal", "cadena", "per_ocu",
             "tipo_vial", "nom_vial", "numero_ext", "nomb_asent", "cod_postal", "cve_mun", "municipio",
             "cve_loc", "localidad", "ageb", "manzana", "latitud", "longitud", "fecha_alta", "nueva_24m"]].reset_index(drop=True)
tiendas["edicion_denue"] = EDICION
print(f"Tiendas {C.ZM_NOMBRE}: {len(tiendas):,}")
print(pd.crosstab(tiendas.canal, tiendas.formato, margins=True), "\n")
pd.crosstab(tiendas.cadena, tiendas.formato, margins=True).sort_values("All", ascending=False)

Tiendas ZM Guadalajara: 24,031
formato      Abarrotes  Minisuper  Supermercado    All
canal                                                 
Moderno             40       1425           157   1622
Tradicional      22306         98             5  22409
All              22346       1523           162  24031 



formato,Abarrotes,Minisuper,Supermercado,All
cadena,,,,
All,22346,1523,162,24031
Independiente,22262,98,5,22365
OXXO,1,933,0,934
7-Eleven,1,192,0,193
Walmart,1,104,73,178
Bodega Aurrera,3,37,10,50
Soriana,1,0,44,45
Six,44,0,0,44
Waldo's,1,23,9,33


Revisión: cadenas detectadas dentro de abarrotes (461110) — verificar que no sean falsos positivos.

In [4]:
tiendas[(tiendas.formato == "Abarrotes") & (tiendas.canal == "Moderno")].groupby("cadena").nom_estab.agg(["count", "first"])

,count,first
cadena,,
7-Eleven,1,SEVEN ELEVEN 7
Bara,2,SUPER BARA
Bodega Aurrera,3,BODEGA AURRERA
Kiosko,2,KIOSKO
Merza / Dax,1,LAGUNITAS
Modelorama,17,ABARROTES MODELORAMA LADA
OXXO,1,TIENDA DE ABARROTES OXXO
Soriana,1,ESTACIONAMIENTO DE SORIANA
Su Super,2,SU SUPER


In [5]:
pd.crosstab(tiendas.municipio, [tiendas.canal, tiendas.formato], margins=True)

canal                           Moderno                        Tradicional                           All
formato                       Abarrotes Minisuper Supermercado   Abarrotes Minisuper Supermercado       
municipio                                                                                               
Acatlán de Juárez                     0         8            1         152         0            0    161
El Salto                              1        48            7        1368         5            0   1429
Guadalajara                          15       439           41        6324        39            3   6861
Ixtlahuacán de los Membrillos         1        13            1         211         2            0    228
Juanacatlán                           0         3            1         138         1            0    143
San Pedro Tlaquepaque                 3       163           21        3352        14            0   3553
Tlajomulco de Zúñiga                  7       219           25        2730         8            0   2989
Tonalá                                2        71            8        2657         9            0   2747
Zapopan                              11       445           51        5047        19            2   5575
Zapotlanejo                           0        16            1         327         1            0    345
All                                  40      1425          157       22306        98            5  24031

## 3. (Opcional) Cruce con OpenStreetMap para detectar tiendas no registradas en DENUE

Se consultan `shop=supermarket` y `shop=convenience` en la ZM. Una tienda OSM a más de 100 m de
cualquier tienda DENUE se marca como *posible tienda no registrada*. Si Overpass no responde, se omite.

In [6]:
USAR_OSM = True
osm_nuevas = pd.DataFrame()
if USAR_OSM:
    s, w, n, e = tiendas.latitud.min() - .02, tiendas.longitud.min() - .02, tiendas.latitud.max() + .02, tiendas.longitud.max() + .02
    q = f"""[out:json][timeout:120];
    (nwr["shop"~"^(supermarket|convenience)$"]({s},{w},{n},{e}););
    out center tags;"""
    for url in ["https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter"]:
        try:
            r = requests.post(url, data={"data": q}, timeout=180,
                              headers={"User-Agent": "analisis-retail-mx/1.0"})  # Overpass exige User-Agent
            r.raise_for_status()
            el = r.json()["elements"]
            break
        except Exception as ex:
            print("Overpass falló en", url, "->", ex.__class__.__name__)
            el = None
    if el:
        osm = pd.DataFrame([{
            "osm_id": f"{x['type']}/{x['id']}", "nombre": x.get("tags", {}).get("name", ""),
            "marca": x.get("tags", {}).get("brand", ""), "shop": x["tags"]["shop"],
            "lat": x.get("lat", x.get("center", {}).get("lat")), "lon": x.get("lon", x.get("center", {}).get("lon")),
        } for x in el])
        from sklearn.neighbors import BallTree
        bt = BallTree(np.radians(tiendas[["latitud", "longitud"]].values), metric="haversine")
        d, _ = bt.query(np.radians(osm[["lat", "lon"]].values), k=1)
        osm["dist_denue_m"] = (d[:, 0] * 6_371_000).round(0)
        osm_nuevas = osm[osm.dist_denue_m > 100].sort_values("dist_denue_m", ascending=False)
        print(f"OSM: {len(osm):,} tiendas | a >100 m de DENUE: {len(osm_nuevas):,}")
osm_nuevas.head(15)

OSM: 919 tiendas | a >100 m de DENUE: 230


,osm_id,nombre,marca,shop,lat,lon,dist_denue_m
458,node/8369494268,Mi Bodega Aurrera,Bodega Aurrera,supermarket,20.376694,-102.929658,21251.0
459,node/8369494272,Oxxo,Oxxo,convenience,20.375705,-102.929987,21229.0
460,node/8369494290,19 Hermanos (Forrajes),,convenience,20.377378,-102.930845,21120.0
463,node/8400642487,Oxxo,Oxxo,convenience,20.707979,-102.956799,11339.0
585,node/12619987011,TIENDA DE ABARROTES ROSITA,,convenience,20.345805,-103.673721,9273.0
590,node/12619987019,TIENDA DE ABARROTES GUILLERMO,,convenience,20.348293,-103.676722,9207.0
573,node/12619986995,TIENDA DE ABARROTES Y PAPELERIA,,convenience,20.347616,-103.675589,9205.0
604,node/12619987049,ABARROTES Y MISCELANEA,,convenience,20.344060,-103.669055,9203.0
581,node/12619987007,TIENDA DE ABARROTES SIN NOMBRE,,convenience,20.346214,-103.672831,9187.0
591,node/12619987020,TIENDA DE ABARROTES GARCIA,,convenience,20.344448,-103.669281,9176.0


## 4. Guardar

In [7]:
tiendas.to_parquet(C.PROC / f"tiendas_denue_{C.SLUG}.parquet", index=False)
for canal in C.CANALES:
    sub = tiendas[tiendas.canal == canal]
    out = C.OUT / f"02_tiendas_{canal.lower()}_{C.SLUG}.xlsx"
    with pd.ExcelWriter(out, engine="openpyxl") as xw:
        sub.to_excel(xw, sheet_name=f"Tiendas {canal}", index=False)
        if canal == "Moderno":
            pd.crosstab(sub.cadena, sub.formato, margins=True).to_excel(xw, sheet_name="Resumen cadenas")
        pd.crosstab(sub.municipio, sub.formato, margins=True).to_excel(xw, sheet_name="Resumen municipio")
        if len(osm_nuevas):
            osm_nuevas.to_excel(xw, sheet_name="OSM no en DENUE", index=False)
    print(f"Guardado: {out.name} ({len(sub):,} tiendas)")

Guardado: 02_tiendas_moderno_zm_guadalajara.xlsx (1,622 tiendas)


Guardado: 02_tiendas_tradicional_zm_guadalajara.xlsx (22,409 tiendas)
